In [0]:
CREATE OR REPLACE TABLE workspace.aq_gold.hourly_wide AS

SELECT
    site_id,
    obs_time,
    obs_date,
    obs_hour,
    year,
    month,

    MAX(
        CASE
            WHEN parameter_code = 'PM2.5'
            THEN value_clean
        END
    ) AS pm25,

    MAX(
        CASE
            WHEN parameter_code = 'PM10'
            THEN value_clean
        END
    ) AS pm10,

    MAX(
        CASE
            WHEN parameter_code = 'NO2'
            THEN value_clean
        END
    ) AS no2,

    MAX(
        CASE
            WHEN parameter_code = 'OZONE'
            THEN value_clean
        END
    ) AS ozone,

    MAX(
        CASE
            WHEN parameter_code = 'TEMP'
            THEN value_clean
        END
    ) AS temp_c,

    MAX(
        CASE
            WHEN parameter_code = 'HUMID'
             AND value_clean BETWEEN 0 AND 100
            THEN value_clean
        END
    ) AS humidity,

    MAX(
        CASE
            WHEN parameter_code = 'WSP'
            THEN value_clean
        END
    ) AS wind_speed,

    MAX(
        CASE
            WHEN parameter_code = 'WDR'
            THEN value_clean
        END
    ) AS wind_dir,

    MAX(
        CASE
            WHEN parameter_code = 'RAIN'
            THEN value_clean
        END
    ) AS rainfall

FROM workspace.aq_dlt.silver_observations

GROUP BY
    site_id,
    obs_time,
    obs_date,
    obs_hour,
    year,
    month;


SELECT COUNT(*) AS hourly_wide_rows
FROM workspace.aq_gold.hourly_wide;

SELECT
    site_id,
    obs_time,
    COUNT(*) AS n
FROM workspace.aq_gold.hourly_wide
GROUP BY
    site_id,
    obs_time
HAVING COUNT(*) > 1;

SELECT
    COUNT(*) AS total_station_hours,

    COUNT(pm25) AS has_pm25,

    COUNT(temp_c) AS has_temp,

    COUNT(humidity) AS has_valid_humidity,

    COUNT(wind_speed) AS has_wind,

    COUNT(rainfall) AS has_rain,

    COUNT(
        CASE
            WHEN pm25 IS NOT NULL
             AND wind_speed IS NOT NULL
            THEN 1
        END
    ) AS pm25_and_wind,

    COUNT(
        CASE
            WHEN pm25 IS NOT NULL
             AND rainfall IS NOT NULL
            THEN 1
        END
    ) AS pm25_and_rain

FROM workspace.aq_gold.hourly_wide;


SELECT
    ROUND(
        100.0 *
        COUNT(
            CASE
                WHEN pm25 IS NOT NULL
                 AND wind_speed IS NOT NULL
                THEN 1
            END
        )
        / COUNT(pm25),
        2
    ) AS pct_pm25_with_wind,

    ROUND(
        100.0 *
        COUNT(
            CASE
                WHEN pm25 IS NOT NULL
                 AND rainfall IS NOT NULL
                THEN 1
            END
        )
        / COUNT(pm25),
        2
    ) AS pct_pm25_with_rain

FROM workspace.aq_gold.hourly_wide;


SELECT
    s.site_id,
    s.site_name,
    s.region,

    COUNT(
        CASE
            WHEN w.pm25 IS NOT NULL
             AND w.wind_speed IS NOT NULL
            THEN 1
        END
    ) AS pm25_wind_hours

FROM workspace.aq_gold.hourly_wide AS w

JOIN workspace.aq_silver.dim_station AS s
    ON w.site_id = s.site_id

GROUP BY
    s.site_id,
    s.site_name,
    s.region

HAVING pm25_wind_hours > 0

ORDER BY pm25_wind_hours DESC;


SELECT
    COUNT(*) AS silver_humidity_rows,
    SUM(CASE WHEN value_clean > 100 THEN 1 ELSE 0 END) AS invalid_humidity_rows
FROM workspace.aq_dlt.silver_observations
WHERE parameter_code = 'HUMID';



SELECT
    COUNT(humidity) AS valid_humidity_in_wide,
    MIN(humidity) AS min_humidity,
    MAX(humidity) AS max_humidity
FROM workspace.aq_gold.hourly_wide;



SELECT
    site_id,
    obs_time,
    pm25,
    pm10,
    no2,
    ozone,
    temp_c,
    humidity,
    wind_speed,
    wind_dir,
    rainfall
FROM workspace.aq_gold.hourly_wide
WHERE pm25 IS NOT NULL
  AND wind_speed IS NOT NULL
ORDER BY obs_time
LIMIT 20;


SELECT
    site_id,
    obs_time,
    COUNT(*) AS n
FROM workspace.aq_gold.hourly_wide
GROUP BY site_id, obs_time
HAVING COUNT(*) > 1;